In [ ]:
import warnings
from copy import deepcopy
from pathlib import Path
import holoviews
import numpy
import pandas
from bokeh import models, palettes, plotting, transform
from panel import Column, Row, Tabs, depends, extension, widgets
from plotly.express import scatter_3d, scatter_matrix
from sklearn.decomposition import KernelPCA
from umap import UMAP
import multiprocessing
from astropy.io import fits
from joblib import Parallel, delayed, parallel_backend

warnings.filterwarnings("ignore", category=UserWarning)

extension()
holoviews.extension("bokeh", "plotly")

In [ ]:
def find_directory(name, init_directory):
    directory_path = Path(init_directory)
    matching_folders = list(directory_path.glob(f"{name}"))
    return matching_folders[0]

def load_fits(file):
    with fits.open(file) as hdul:
        img = hdul[0].data
    return img

def parallel_with_one_arg(func, data):
    nb_cpus = multiprocessing.cpu_count()
    print(f"using {nb_cpus} cpus")
    with parallel_backend("threading", n_jobs=nb_cpus):
        return numpy.array(
            Parallel(prefer="threads", verbose=0)(delayed(func)(arg=arg) for arg in data)
        )

def load_dataset(name_db, thresh):
    print("loading images...", end="")
    list_images = sorted(name_db.glob("simu_*.fits"))
    simu = parallel_with_one_arg(func=lambda arg: load_fits(arg), data=list_images)
    print("...ok")

    print("loading shear...", end="")
    list_shear = sorted(name_db.glob("shear_*.fits"))
    shear = parallel_with_one_arg(func=lambda arg: load_fits(arg), data=list_shear)
    print("...ok")

    print("loading csv...", end="")
    inputs = pandas.read_csv(name_db.parent / f"{name_db.name}_hypercube.csv", delimiter=",")
    galaxy_parameters = numpy.array(
        pandas.read_csv(name_db.parent / f"{name_db.name}_galaxy_parameters.csv", delimiter=",")
    )
    print("...ok")

    if thresh > 0:
        unique_random_integers = numpy.random.choice(
            numpy.arange(inputs.shape[0]), thresh, replace=False
        )
        database = {
            "inputs": inputs.iloc[unique_random_integers],
            "simu": simu[unique_random_integers],
            "shear": shear[unique_random_integers],
        }
        database["inputs"] = database["inputs"].reset_index(drop=True)
    else:
        database = {"inputs": inputs, "simu": simu, "shear": shear}

    number_of_sample = database["inputs"].shape[0]
    number_of_inputs = database["inputs"].shape[1]
    print(f"number of samples: {number_of_sample} ")
    print(f"number of inputs: {number_of_inputs, list(database['inputs'].keys())}")
    print("Number of NaNs:", numpy.isnan(database["simu"]).sum())
    return database, galaxy_parameters


In [ ]:
FOLDER_TEST = "v0/validation"
folder = find_directory(FOLDER_TEST, "/net/GECO/pisco/PIX2SHEAR/DATASETS_EUCLID")

In [ ]:
database, galaxy_parameters = load_dataset(folder, 0)

# kpca

In [ ]:
copy_database = deepcopy(database)
reshaped_database = numpy.reshape(
    copy_database["simu"], (numpy.shape(copy_database["simu"])[0], -1)
)
del copy_database

In [ ]:
NB_PRINC_CPNTS = 3

KPCA_total = KernelPCA(n_components=NB_PRINC_CPNTS, kernel="rbf")

embedding_kpca = KPCA_total.fit_transform(reshaped_database)
print(embedding_kpca.shape)

title_columns = [f"PC{i+1}" for i in range(NB_PRINC_CPNTS)]
dataframe_embedding_kpca = pandas.DataFrame(embedding_kpca, columns=title_columns)

In [ ]:
columns_to_keep = ["PC1", "PC2", "PC3"]
selected_data = dataframe_embedding_kpca[columns_to_keep]

In [ ]:
def create_scatter_matrix(selected_data):
    colored_values = selected_data.columns.to_list()
    color_selector = widgets.Select(name="Colored", options=colored_values, value=colored_values[0])
    colors = ["Viridis", "Plasma", "Cividis", "Jet"]
    color_scale_selector = widgets.Select(name="Color", options=colors, value=colors[0])

    @depends(color=color_selector, color_scale=color_scale_selector)
    def update_plot(color, color_scale):
        return scatter_matrix(
            selected_data,
            dimensions=colored_values,
            color=color,
            color_continuous_scale=color_scale,
            width=1200,
            height=1200,
        )

    return Column(Row(color_selector, color_scale_selector), update_plot)

In [ ]:
Tabs(("KERNEL PCA plot", create_scatter_matrix(selected_data)))

# UMAP

In [ ]:
def create_umap_plot(inputs, reshaped_db):
    nb_neighbors_input = widgets.IntInput(name="Number of Neighbors", value=2, step=1)
    bool_selector = widgets.Select(name="New random State", options=["Yes", "No"], value="No")
    random_state_slider = widgets.IntSlider(name="Seed", start=0, end=100, value=13)
    bool_selector_class = widgets.Select(
        name="Precise 2 classes for Umap", options=["Yes1", "Yes2", "No"], value="Yes1"
    )
    color_options = list(inputs.keys())
    color_selector = widgets.Select(name="Color", options=color_options, value=color_options[0])

    # interactive objects
    @depends(
        color=color_selector,
        nb_neighbors=nb_neighbors_input,
        bool_rd=bool_selector,
        bool_class=bool_selector_class,
        seed=random_state_slider,
    )
    def update_plot(color, nb_neighbors, bool_rd, bool_class, seed):
        # compute umap
        if bool_rd == "No":
            seed = 13
        embedding = UMAP(
            n_neighbors=nb_neighbors,
            n_components=2,
            init="pca",
            random_state=seed,
            learning_rate=0.8,
        )
        # embedding = UMAP(n_neighbors=nb_neighbors, n_components=2, init='random', random_state=seed, metric='correlation', learning_rate=0.5)
        if bool_class == "No":
            umap_results = embedding.fit_transform(reshaped_db)
        elif bool_class == "Yes1":
            umap_results = embedding.fit_transform(X=reshaped_db, y=database["inputs"]["g1"])
        elif bool_class == "Yes2":
            umap_results = embedding.fit_transform(X=reshaped_db, y=database["inputs"]["g2"])

        # Bokeh data source for results
        column_data = dict(x_umap=umap_results[:, 0], y_umap=umap_results[:, 1])
        for i, key in enumerate(inputs.keys()):
            column_data[f"{key}"] = inputs.to_numpy()[:, i]
        source = models.ColumnDataSource(data=column_data)

        # figure
        p_umap = plotting.figure(
            width=1000,
            height=1000,
            title="UMAP Results",
            tools="lasso_select,reset,pan,wheel_zoom,save,tap",
        )
        mapper = transform.linear_cmap(
            field_name=color,
            palette=palettes.Viridis256,
            low=min(column_data[color]),
            high=max(column_data[color]),
        )
        p_umap.circle("x_umap", "y_umap", source=source, size=6, fill_alpha=0.7, color=mapper)
        p_umap.add_layout(
            models.ColorBar(
                color_mapper=models.LinearColorMapper(palette=palettes.Viridis256),
                width=8,
                location=(0, 0),
            ),
            "right",
        )
        p_umap.xgrid.grid_line_color = None
        p_umap.ygrid.grid_line_color = None
        p_umap.background_fill_color = "white"
        p_umap.xaxis.major_label_text_font_size = "0pt"
        p_umap.yaxis.major_label_text_font_size = "0pt"

        # add the hover tool
        hover_umap = models.HoverTool()
        hover_umap.tooltips = [(f"{key}", f"@{key}") for _, key in enumerate(inputs.keys())]
        p_umap.add_tools(hover_umap)

        # linked tableau
        columns = [
            models.TableColumn(field=f"{key}", title=f"{key}")
            for _, key in enumerate(inputs.keys())
        ]
        data_table = models.DataTable(source=source, columns=columns, width=800, height=400)

        return Row(p_umap, data_table)

    # set up
    return Column(
        Row(
            color_selector,
            bool_selector_class,
            nb_neighbors_input,
            bool_selector,
            random_state_slider,
        ),
        update_plot,
    )


Tabs(("UMAP plot", create_umap_plot(database["inputs"], reshaped_database)))

In [ ]:
def create_umap3d_plot(inputs, reshaped_db):
    color_maps = ["Viridis", "Plasma", "Cividis", "Jet"]
    color_scale_selector = widgets.Select(
        name="Color Style", options=color_maps, value=color_maps[3]
    )
    color_options = list(inputs.keys())
    color_selector = widgets.Select(name="Color", options=color_options, value=color_options[0])
    nb_neighbors_input = widgets.IntInput(name="Number of Neighbors", value=2, step=1)
    bool_selector_state = widgets.Select(name="New random State", options=["Yes", "No"], value="No")
    bool_selector_class = widgets.Select(
        name="Precise 2 classes for Umap", options=["Yes", "No"], value="Yes"
    )
    random_state_slider = widgets.IntSlider(name="Seed", start=0, end=100, value=13)

    @depends(
        color=color_selector,
        color_scale=color_scale_selector,
        nb_neighbors=nb_neighbors_input,
        bool_rd=bool_selector_state,
        bool_class=bool_selector_class,
        seed=random_state_slider,
    )
    def update_plot(color, color_scale, nb_neighbors, bool_rd, bool_class, seed):
        if bool_rd == "No":
            seed = 13
        embedding = UMAP(
            n_neighbors=nb_neighbors,
            n_components=3,
            init="pca",
            random_state=seed,
            learning_rate=0.2,
        )
        # embedding = UMAP(n_neighbors=nb_neighbors, n_components=3, init='random', random_state=seed, learning_rate=0.8)
        if bool_class == "No":
            umap_results = embedding.fit_transform(reshaped_db)
        else:
            umap_results = embedding.fit_transform(X=reshaped_db, y=inputs["g1"])

        fig = scatter_3d(
            umap_results,
            x=0,
            y=1,
            z=2,
            color=inputs[f"{color}"],
            color_continuous_scale=color_scale,
            width=1000,
            height=1000,
        )
        fig.update_traces(marker=dict(size=3))
        fig.update_layout(
            scene=dict(xaxis_title="UMAP-1", yaxis_title="UMAP-2", zaxis_title="UMAP-3"),
            coloraxis_colorbar=dict(title=f"{color}"),
        )
        return fig

    return Column(
        Row(nb_neighbors_input, bool_selector_state, random_state_slider, bool_selector_class),
        update_plot,
    )


Tabs(("UMAP plot", create_umap3d_plot(database["inputs"], reshaped_database)))